# Sampling flux vacua

**What's in this notebook?** This notebook demonstrates how to efficiently sample large numbers of SUSY flux vacua using the `FluxEFT.sample_SUSY_flux_vacua` wrapper. It builds on the ISD sampling concepts from notebook [4](../01_basics/04_sampling_module.ipynb) and the Newton refinement approach from notebook [7](06_ISD_sampling_principle.ipynb).

**In this notebook, you will learn:**
- How to use `model.sample_SUSY_flux_vacua` for automated large-scale sampling
- Running with a single optimiser (`mode="Fflux"`) vs. multiple optimisers in sequence
- Traditional uniform flux sampling (`mode="random"`) and its limitations compared to ISD sampling
- Constrained sampling: restricting to specific regions of moduli space or requiring small $|W_0|$
- Post-processing: checking GV convergence with the ratio $|F_{\mathrm{inst}}|/|F_{\mathrm{poly}}|$

**Workflow summary:**

| Step | Code | Description |
|------|------|-------------|
| 1. Model | `model = jvc.FluxEFT(h12=2, ...)` | Load or construct the geometry |
| 2. Sampler | `sampler = jvc.data_sampler(model, ...)` | Configure flux and moduli bounds |
| 3. Optimiser | `vmapping_func(model.linearised_shifts, mode=...)` | Build vmapped Newton solver |
| 4. Sample | `model.sample_SUSY_flux_vacua(N=..., ...)` | Run the sampling loop |
| 5. Post-process | `model.F_inst(moduli) / model.F_LCS_poly(moduli)` | Check GV convergence |

**Related notebooks:** [04_sampling_module](../01_basics/04_sampling_module.ipynb) (sampling infrastructure), [06_ISD_sampling_principle](06_ISD_sampling_principle.ipynb) (single-vacuum walkthrough), [9_sampling_vacua_from_fluxes](9_sampling_vacua_from_fluxes.ipynb) (flux-first sampling approach), [10b_stochastic_flux_search](../03_flux_bounding/10b_stochastic_flux_search.ipynb) (stochastic flux enumeration).

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

In [ ]:
# General imports
import warnings, sys, time
import numpy as np
from tqdm.auto import tqdm

# JAX imports
from jax import jit, vmap
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc
from jaxvacua.util import vmapping_func

warnings.filterwarnings('ignore')

## Plotting function

In [ ]:
sys.path.append("../")
from plot_utils import make_overview_plots

## Sampling SUSY flux vacua at $h^{1,2}=2$

Load model from files

In [ ]:
#if False:
if True:
    h12=2
    model = jvc.FluxVacuaFinder(h12 = h12, Q=276, model_ID = 1, maximum_degree = 2,limit="LCS",model_type="KS")

Alternatively, use CYTools

In [ ]:
from cytools import Polytope, Cone, fetch_polytopes, read_polytopes

if False:
    p = fetch_polytopes(h11=2,h12=272,limit=5,lattice="N",as_list=True)[0]
    cy = p.triangulate().get_cy()
    mcap = cy.mori_cone_cap(in_basis=True)
    Kcup = mcap.dual_cone()
    basis_trafo = Kcup.extremal_rays()

    model = jvc.FluxVacuaFinder(h12=cy.h11(), Q=cy.h11()+cy.h12()+2, model_type="KS", maximum_degree=2, 
                                      use_cytools=True, mirror_cy = cy, basis_transformation=basis_trafo)

As objective function, we take the $F$-terms for the moduli as computed in `model.DW`.
For later purposes, we vectorise this function by using `jax.vmap`:

In [ ]:
DW_v = vmap(model.DW)

It is also convenient to introduce a data sampler that constrains our sampling procedure to specific regions in moduli and flux space:

In [ ]:
sampler = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[0,5],axion_bounds=[-0.5,0.5])

For reproducability, we use a seed for random number generation

In [ ]:
seed = 42
rns_key = jvc.PRNGSequence(seed)

Initial guesses can then be obtained as follows:

In [ ]:
sampler.update_interior_points(num_pts = 10000)
moduli,tau,fluxes = sampler.initial_guesses(5,rns_key=rns_key)
moduli,tau,fluxes

### ISD sampling with a single optimiser

We start by defining a numerical optimiser.
In principle, any optimiser could in principle do the job provided it takes as inputs `moduli,tau,fluxes` and gives the following outputs:

```updated_moduli,updated_tau,updated_fluxes,checks = optimiser(moduli,tau,fluxes)```

Here, `checks` is an array containing boolean values that specify if the updated values satisfy all consistency conditions (like staying inside the mirror Kähler cone or alike).

There are a couple of pre-implemented optimisation algorithms based on the results of [??](??) that we can use.
Here we use the $F$-sampling algorithm:

In [ ]:
kwargs = {"Q":model.D3_tadpole,"return_flag":True,"constraints": None,"remove_NANs":True,"in_axes":(0,0,0)}

linearised_shifts_F = vmapping_func(model.linearised_shifts,mode="Fflux",**kwargs)


We sample minima by running the following code with `optimiser=linearised_shifts_F` and `objective_fct=DW_v`:

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_flux_vacua(N=10**2,
                                                         rns_key=rns_key,
                                                         sampler=sampler,
                                                         optimiser=linearised_shifts_F,
                                                         objective_fct=DW_v,
                                                         vmap_dim=10,
                                                         print_progress=True)

By increasing the dimension used for vectorisation `vmap`,
we can speed up the sampling

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_flux_vacua(N=10**3,
                                                         rns_key=rns_key,
                                                         sampler=sampler,
                                                         optimiser=linearised_shifts_F,
                                                         objective_fct=DW_v,
                                                         vmap_dim=10**2,
                                                         print_progress=True)

Let us plot the results

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.5,0.5],[0,10]],tau_range=[[-0.5,0.5],[0,10]],W0_range=[-10,10])

### Running with multiple optimisers

We can also run the above code with multiple optimisers

In [ ]:
kwargs = {"Q":model.D3_tadpole,"return_flag":True,"constraints": None,"remove_NANs":True,"in_axes":(0,0,0)}

linearised_shifts_ISD = vmapping_func(model.linearised_shifts,mode="ISD",**kwargs)
linearised_shifts_H = vmapping_func(model.linearised_shifts,mode="Hflux",**kwargs)
linearised_shifts_F = vmapping_func(model.linearised_shifts,mode="Fflux",**kwargs)

optimisers = [linearised_shifts_ISD,linearised_shifts_H,linearised_shifts_F]

We can provide a list as `optimisers=optimisers` and run

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_flux_vacua(N=10**2,
                                                           rns_key=rns_key,
                                                           sampler=sampler,
                                                           optimisers=optimisers,
                                                           objective_fct=DW_v,
                                                           vmap_dim=10**2,
                                                           print_progress=True)

Let us plot the solutions obtained

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.5,0.5],[0,10]],tau_range=[[-0.5,0.5],[0,10]],W0_range=[-10,10])

### Old method: uniform sampling of fluxes from box

In many applications, it has been common practice to search for flux vacua by sampling fluxes from a uniform distribution
defined on some hypercube in flux space.
This method can also be employed here by using `mode="random"` as keyword argument when defining the optimiser:

In [ ]:
kwargs = {"Q":model.D3_tadpole,"return_flag":True,"constraints": None,"remove_NANs":True,"in_axes":(0,0,0)}

linearised_shifts = vmapping_func(model.linearised_shifts,mode="random",**kwargs)

We run the same function as above just with a new optimiser

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_flux_vacua(N=2*10**2,
                                                           rns_key=rns_key,
                                                           sampler=sampler,
                                                           optimiser=linearised_shifts,
                                                           moduli_sampling_mode="cone",
                                                           objective_fct=DW_v,
                                                           vmap_dim=10**2,
                                                           print_progress=True)

We visualise the solutions here

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.5,0.5],[0,5]],tau_range=[[-0.5,0.5],[0,2]],W0_range=[-10,10])

As discussed e.g. in [2306.06160](https://arxiv.org/abs/2306.06160) and [2308.15525](https://arxiv.org/abs/2308.15525),
the resulting distribution of moduli clusters around the boundary of the mirror Kähler cone.
More specifically, since we take the leading order mirror dual worldsheet instanton contributions into account,
the vacua are typically shifted away from the walls of the mirror Kähler cone towards the region of GV convergence.
In practice, we should check whether our solutions are under control by e.g. imposing that the ratio
\begin{equation}
    \dfrac{|F_{\mathrm{inst}}|}{|F_{\mathrm{poly}}|} <\epsilon
\end{equation}
for some $\epsilon\ll 1$. For $\epsilon=0.1$, this is done as follows:

In [ ]:
Finst = jax.vmap(model.F_inst)(moduli)
Fpoly = jax.vmap(model.F_LCS_poly)(moduli)
epsilon = 0.1
flag = np.abs(Finst)/np.abs(Fpoly)<epsilon
moduli_res,tau_res,fluxes_res = moduli[flag],tau[flag],fluxes[flag]
print(f"#vacua: {moduli.shape[0]}  without post-selection")
print(f"#vacua: {moduli_res.shape[0]}  with those vacua outside naive radius of convergence removed  ")

In [ ]:
make_overview_plots(model,moduli_res,tau_res,fluxes_res,moduli_range=[[-0.5,0.5],[0,5]],tau_range=[[-0.5,0.5],[0,2]],W0_range=[-10,10])

### Constrained sampling

There is also the option to include restrictions on the vacua by imposing desired properties.
In many application, it might be desirable for moduli to take values in a certain range
or other quantities like $g_s$, $N_{\mathrm{flux}}$ or $W_0$ to satisfy specific bounds.
Below, we present different two different use cases, but generalisations are straight forward.

Before we start, let us stress that, when making use of ISD sampling, the methods presented below
**are not** just rejection sampling. Instead, as described in [upcoming](upcoming), when momentarily working
with continuous ISD fluxes, various quantities like the moduli VEVs or $W_0$ can be well approximated
**before minimisation**. This allows for a much more efficient way to obtain flux vacua with constraints.

#### Regions of moduli space

Here, as a proof of concept, we would like the moduli to satisfy $2< Im(z^i)<3$ and the axio-dilaton $Im(\tau)>4$.
This is achieved by defining a function that takes as input `moduli,tau,flux` and returns a boolean value:

In [ ]:
@jit
def constraints_model(moduli,tau,flux):
    
    return ((jnp.all(jnp.imag(moduli)<3.,axis=0))&(jnp.all(jnp.imag(moduli)>2.,axis=0))&(jnp.imag(tau)>4))


kwargs = {"Q":model.D3_tadpole,"return_flag":True,"constraints": constraints_model,"remove_NANs":True,"in_axes":(0,0,0)}

linearised_shifts_F_v = vmapping_func(model.linearised_shifts,mode="Fflux",**kwargs)

Let us redefine our sample to accomodate these constraints

In [ ]:
sampler = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[2,3],dilaton_bounds=[4,10])

We sample minima by running the following code with `optimiser=linearised_shifts_F_v` and `objective_fct=DW_v`:

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_flux_vacua(N=10**2,
                                                         rns_key=rns_key,
                                                         sampler=sampler,
                                                         optimiser=linearised_shifts_F_v,
                                                         objective_fct=DW_v,
                                                         vmap_dim=10**2,
                                                         print_progress=True)

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.5,0.5],[1,4]],tau_range=[[-0.5,0.5],[3,12]],W0_range=[-10,10])

#### Vacua with small $|W_0|$

Another application concerns finding flux vacua with small $|W_0|$.
The code below implements the algorithm described in [upcoming](upcoming) for general constraints.
The idea is simple: when using ISD sampling for the fluxes, we can momentarily work with continuous fluxes
and estimate values of certain quantities like $|W_0|$ before finding the true $F$-term minimum with quantised fluxes.
This provides in many cases a very good approximation of the true value of said quantity in the actual vacuum.
The advantage is that the expensive minimisation step on uninteresting vacua can be avoided and one simply has to sample
efficiently. **REWRITE**

In practice, this works by providing a refined constraint function `check_constraints`

**THIS IS STILL WORK IN PROGRESS**

In [ ]:
@jit
def check_constraints(moduli,tau,flux):
    
    W0 = model.superpotential_gauge_invariant(moduli,tau,flux)
    
    return (jnp.abs(W0)<0.1)#&(jnp.all(moduli.imag>1,axis=0))

kwargs = {"Q":model.D3_tadpole,"return_flag":True,"constraints":check_constraints,"remove_NANs":True,"in_axes":(0,0,0)}

We define a sampling object

In [ ]:
seed = 42
rns_key = jvc.PRNGSequence(seed)

sampler = jvc.data_sampler(model,flux_bounds=[-3,3],moduli_bounds=[1,3],dilaton_bounds=[1,10])

We define the objective function and the optimisation steps

In [ ]:
DW_v = vmap(model.DW)

linearised_shifts_ISD_v = vmapping_func(model.linearised_shifts,mode="ISD",**kwargs)
linearised_shifts_H_v = vmapping_func(model.linearised_shifts,mode="Hflux",**kwargs)
linearised_shifts_F_v = vmapping_func(model.linearised_shifts,mode="Fflux",**kwargs)
optimisers = [linearised_shifts_ISD_v,linearised_shifts_H_v,linearised_shifts_F_v]

In [ ]:
optimisers = [linearised_shifts_H_v]

In [ ]:
optimisers = [linearised_shifts_F_v]

We then run the same code as above and simply provide the constraint function as follows
`constraints = vmap(check_constraints,in_axes=(0,0,0))`:

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_flux_vacua(N=10**2,
                                                           rns_key=rns_key,
                                                           constraints = vmap(check_constraints,in_axes=(0,0,0)),
                                                           moduli_sampling_mode="cone",
                                                           sampler=sampler,
                                                           optimisers=optimisers,
                                                           objective_fct=DW_v,
                                                           vmap_dim=10**2,
                                                           print_progress=True)

Let us plot the final solutions

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.5,0.5],[1,4]],tau_range=[[-0.5,0.5],[3,12]],W0_range=[-10,10])

## Summary

This notebook demonstrated automated large-scale sampling of SUSY flux vacua using `model.sample_SUSY_flux_vacua`. Key takeaways:

- The `vmapping_func` helper wraps `model.linearised_shifts` into a batched Newton step; `mode=` selects the optimisation strategy.
- Passing a list `optimisers=[...]` applies them in sequence, improving coverage.
- Uniform flux sampling (`mode="random"`) is simple but produces vacua clustered at the Kähler cone boundary.
- ISD-based optimisers (`"Fflux"`, `"Hflux"`, `"ISD"`) give much better moduli coverage.
- The GV convergence ratio $|F_\mathrm{inst}|/|F_\mathrm{poly}|$ provides a quick sanity check on vacuum quality.
- Constraint functions allow targeted searches (moduli regions, small $|W_0|$) without pure rejection sampling.

**Next steps:**
- [9_sampling_vacua_from_fluxes](9_sampling_vacua_from_fluxes.ipynb) — flux-first approach: enumerate vacua for given input fluxes
- [10_flux_bounding](../03_flux_bounding/10_flux_bounding.ipynb) — enumerate all flux vacua within a bounded tadpole region
- [10b_stochastic_flux_search](../03_flux_bounding/10b_stochastic_flux_search.ipynb) — stochastic search over the flux lattice